In [161]:
import os
from pathlib import Path

import pandas as pd

from cc import load_df, merge_left, limited_metrics, full_metrics


# Screening Consistency Checks

This notebook mirrors the style of `coding.ipynb`, but for the single-label screening task.

## Checks included

| Check | Estimate (left) | Truth (right) | Metrics |
|---|---|---|---|
| **CC1** | `s1_abs` | `s1_full` | `limited_metrics` |
| **CC2a** | `s1_llm` | `s1_full` | `limited_metrics` |
| **CC2b** | `s2_llm` | `s2_abs` | `full_metrics` |

## Interpretation

- **CC1**: human baseline for how well abstract-only manual labels recover full-text train labels.
- **CC2a**: how well LLM labels recover full-text labels for the provided split.
- **CC2b**: how well LLM labels recover author/manual abstract labels on the mixed `search-sample` set.

The join is always **left JOIN right on record ID**. The left side determines which records are evaluated.


In [162]:
RUN_NAME = "l0_cc2a_b_test_170426_f3"
MODE = "test"  # "train" | "dev" | "test"

S1_LLM_PATH      = "../data/labels/eval/l0_cc2a_test_170426_f3/data/screening.xlsx"
S2_LLM_PATH      = "../data/labels/eval/l0_cc2b_170426_f3/data/screening.xlsx"


In [163]:
# Screening dataset paths
S_ANNOTATED_PATH = "../data/consistency-check-datasets/screening/annotated_screening_dataset.xlsx"
S1_ABS_SHEET     = "train-sample"
S2_ABS_SHEET     = "search-sample"

# Full-text screening labels
S1_FULL_PATH     = f"../data/labels/l0/{MODE}/L0) Screening.csv"


_results = []

_check_sources = {
    "CC1: S1 Abstract vs S1 Full-text": {
        "truth": S1_FULL_PATH,
        "estimate": f"{S_ANNOTATED_PATH} [{S1_ABS_SHEET}]",
    },
    "CC2a: S1 LLM vs Full-text": {
        "truth": S1_FULL_PATH,
        "estimate": S1_LLM_PATH,
    },
    "CC2b: S2 LLM vs S2 Abstract": {
        "truth": f"{S_ANNOTATED_PATH} [{S2_ABS_SHEET}]",
        "estimate": S2_LLM_PATH,
    },
}


## CC1
**Manual abstract annotations (`s1_abs`) vs full-text train labels (`s1_full`)**

Truth: `s1_full`  
Estimate: `s1_abs`  
Join: `s1_abs` LEFT JOIN `s1_full`


In [164]:
if MODE == "train":
    left = load_df(
        S_ANNOTATED_PATH,
        sheet=S1_ABS_SHEET,
        id_col="custom_id",
        label_col="eligibility",
    )

    right = load_df(
        S1_FULL_PATH,
        id_col="UT (Unique WOS ID)",
        label_col="eligbility",
    )

    m1 = merge_left(left, right)
    summary = limited_metrics(m1["y_true"], m1["y_pred"])
    display(summary)
    _results.append(("CC1: S1 Abstract vs S1 Full-text", summary))
else:
    print("Skipping CC1: available only when MODE == 'train'.")


Skipping CC1: available only when MODE == 'train'.


## CC2a
**LLM screening (`s1_llm`) vs full-text labels (`s_full`)**

Truth: `s_full`  
Estimate: `s1_llm`  
Join: `s1_llm` LEFT JOIN `s_full`

> Use `MODE` and `S1_LLM_PATH` to switch between train/dev/test runs.


In [165]:
left = load_df(
    S1_LLM_PATH,
    id_col="custom_id",
    label_col="pred_label",
)

right = load_df(
    S1_FULL_PATH,
    id_col="UT (Unique WOS ID)",
    label_col="eligbility",
)

m2 = merge_left(left, right)
summary = limited_metrics(m2["y_true"], m2["y_pred"])
display(summary)
_results.append(("CC2a: S1 LLM vs Full-text", summary))


  Matched: 482


,Metric,Value
0,Records evaluated,482.0000
1,Observed Agreement,0.8921
2,PABAK,0.7842
3,Recall (ELIGIBLE),0.8921
4,False Negative Rate,0.1079
5,TP,430.0000
6,FN,52.0000


## CC2b
**LLM screening on the search sample (`s2_llm`) vs manual abstract labels (`s2_abs`)**

Truth: `s2_abs`  
Estimate: `s2_llm`  
Join: `s2_llm` LEFT JOIN `s2_abs`

Because `search-sample` is a mixed eligible/ineligible set, this check uses `full_metrics`.


In [166]:
if S2_LLM_PATH and Path(S2_LLM_PATH).exists():
    left = load_df(
        S2_LLM_PATH,
        id_col="custom_id",
        label_col="pred_label",
    )

    right = load_df(
        S_ANNOTATED_PATH,
        sheet=S2_ABS_SHEET,
        id_col="custom_id",
        label_col="eligibility",
    )

    m3 = merge_left(left, right)
    summary = full_metrics(m3["y_true"], m3["y_pred"])
    display(summary)
    _results.append(("CC2b: S2 LLM vs S2 Abstract", summary))
else:
    print("Skipping CC2b: set S2_LLM_PATH to the search-sample screening predictions file.")


  Matched: 400


,Metric,Value
0,Records evaluated,400.0000
1,Observed Agreement,0.8525
2,Cohen's Kappa,0.6783
3,PABAK,0.7050
4,TP,110.0000
5,FP,48.0000
6,FN,11.0000
7,TN,231.0000
8,Precision (ELIGIBLE),0.6962
9,Recall (ELIGIBLE),0.9091


# Output


In [167]:
OUTPUT_DIR = f"../data/labels/eval/consistency-check/screening/{RUN_NAME}"
OUTPUT_XLSX = f"{OUTPUT_DIR}/screening_results.xlsx"

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not _results:
    print("Nothing to save")
else:
    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        sheet = "Screening"
        row = 0
        for check_name, metrics_df in _results:
            pd.DataFrame([[f"── {check_name} ──"]]).to_excel(
                writer, sheet_name=sheet, startrow=row, index=False, header=False
            )
            row += 1
            src = _check_sources.get(check_name, {})
            if src:
                pd.DataFrame([
                    [f"Truth:    {src['truth']}"],
                    [f"Estimate: {src['estimate']}"],
                ]).to_excel(writer, sheet_name=sheet, startrow=row, index=False, header=False)
                row += 2
            metrics_df.to_excel(writer, sheet_name=sheet, startrow=row, index=False)
            row += len(metrics_df) + 3
    print(f"Saved - {OUTPUT_XLSX}")


Saved - ../data/labels/eval/consistency-check/screening/l0_cc2a_b_test_170426_f3/screening_results.xlsx
